# Exercise 2 - 184.702 Machine Learning (VU 3,0) 2025W
## Description
**TASK**

Implement a regression tree algorithm and a random forest algorithm (based on the implemented regression tree algorithm) for predicting numeric values– You can find various implementations for these algorithms. However, we can also apply our own ideas for splitting of instances
+ We should implement these algorithms from scratch (not using any part of existing code)
+ We can use existing code/functions for general parts like: Code for reading the input and testing the algorithm (cross- validation, performance metrics for regression...) 

**COMPARISON**

Compare the implemented techniques with the existing implementations of regression trees/random forest and one other existing regression techniques (we may use the default parameters for the existing techniques) 

+ Experiment with at least three configurations (number of trees, ...) for random 
forest
+ Using at least two performance metrics for comparison
+ Applying cross-validation

***Conclusions***
+ How efficient are our algorithms?
+ Performance of our algorithms?
+ Other findings


# RegressionTree Implementation


## Code section

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor


from typing import Literal, Dict, Any, Self
from numpy.typing import ArrayLike

import logging

logging.basicConfig(level=logging.WARNING,
                    format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger()
logger.setLevel(logging.WARNING)

The principle of building a Regression Tree follows the same approach as the creation of a Classification Tree.

We search for the feature which splits the target feature values most purely, divide the dataset along the values of this descriptive feature and repeat this process for each of the sub datasets until we accomplish a stopping criteria. If we accomplish a stopping criteria, we grow a leaf node.

Most notable difference is when to stop:
If we now consider the property of our new continuously scaled target feature we mention that the third stopping criteria can no longer be used since the target feature values can now take on an infinite number of different values. Consequently, it is most likely that we will not find pure target feature values until there is only one instance left in the dataset.

Long story short, there is in general nothing like pure target feature values.

To address this issue, we will introduce an early stopping criteria that returns the average value of the target feature values left in the dataset if the number of instances in the dataset is e.g. <= 5.

In [ ]:
class RegressionTreeNico():
    '''
    Custom implementation of a regression decision tree for 
    184.702 Machine Learning (VU 3,0) 2025W

    This class builds a regression tree from scratch and supports both
    categorical and continuous features. It uses variance/MSE reduction
    as the splitting criterion. It also supports recursive tree growth with
    configurable stopping criteria (minimum instances per leaf, maximum depth).

    '''    

    def __init__(self) -> None:
        pass

    def _calc_MSE(self, Y_true: ArrayLike, Y_pred: ArrayLike) -> np.float64:
        '''
        Calculates mean squared error

        Returns:
            MSE of given features
        
        '''
        return np.square(np.subtract(Y_true, Y_pred)).mean()

    def _calc_variance_cat_features(self, data: ArrayLike, target_name: str, which_feature_name: str) -> np.float64:
        '''
        Used to calculate the variance of categorical features to decide which feature to use for the next leaf via minimum variance!

        Parameters
        ----------
        data : ArrayLike
            The data set in which we want to calculate the variance for a given feature
        target_name: str
            Name of the target feature
        which_feature_name: str
            Name of the feature for which we want to calculate the variance.

        Returns
        -------
        np.float64
            The (weighted) variance of the feature in the given data
        '''
        
        # prep building blocks for vectorized variance calculation
        grouped_data = data.groupby(which_feature_name)[target_name]
        sizes = grouped_data.size()
        
        # In case we have only one appearance of a value (size N = 1), 
        # then we have to make sure it makes 0 instead of inf because of the division / (N - 1)
        feature_variances = grouped_data.var(ddof=1).fillna(0) 

        weighted_feature_variances = (sizes / len(data)) * feature_variances

        # return the total weighted variance for the feature
        return weighted_feature_variances.sum()
    
        # old non-vectorized part
        feature_values = np.unique(data[which_feature_name])
        feature_variance = 0

        for value in feature_values:
            subset = data[data[which_feature_name] == value].reset_index()

            # In case we have only one appearance of a value, then we have to make sure it makes 0 instead of inf because of the division / (N - 1)
            if len(subset) <= 1:
                subset_var = 0.0
            else:
                # standard in np.var is divided by N instead of N - 1!
                subset_var = (len(subset)/len(data)) * np.var(subset[target_name], ddof=1)

            feature_variance += subset_var
        
        return feature_variance
    
    def _calc_variance_cont_features(self, data: ArrayLike, target_name: str, which_feature_name: str) -> np.float64:
        '''
        Used to calculate the variance of continous features to decide which feature to use for the next leaf via minimum variance!
        
        Parameters
        ----------
        data : ArrayLike
            The data set in which we want to calculate the variance for a given feature
        target_name: str
            Name of the target feature
        which_feature_name: str
            Name of the feature for which we want to calculate the variance.

        Returns
        -------
        np.float64
            The (weighted) variance of the feature in the given data
        '''   

        # initialize weighted mse
        best_weighted_mse = np.inf
        best_threshold = np.inf
                    
        # sort the data by the feature
        data = data.sort_values(by=which_feature_name)
        X = data[which_feature_name].values
        y = data[target_name].values

        # now do all the things we did in the loop in one step!
        # summing up enables us to calculate for each i in a loop to have the sums/means already:
        
        # left side 
        cum_sum = np.cumsum(y)
        cum_squared_sum = np.cumsum(y**2)
        counts = np.arange(1, len(y) + 1) 
        
        # e.g. for i = 4
        # cum_sum = sum up until 4th row
        # counts = well.. four

        # mse can also be written (after rearranging) as:
        # 1/N * sum_over_i(y_i**2) - mean**2
        left_mean = cum_sum[:-1] / counts[:-1] # -1: everything but the last as this would be the total sum
        left_mse = (cum_squared_sum[:-1] / counts [:-1]) - left_mean**2 
        
        # total values as prep for right side 
        total_sum = cum_sum[-1]
        total_squared_sum = cum_squared_sum[-1]
        total_count = len(y)

        # right side
        right_sum = total_sum - cum_sum[:-1]
        right_sq_sum = total_squared_sum - cum_squared_sum[:-1]
        right_count = total_count - counts[:-1]

        right_mean = right_sum / right_count
        right_mse = (right_sq_sum / right_count) - right_mean**2

        # weighted MSE for all the different splits
        weighted_mse = (counts[:-1] * left_mse + right_count * right_mse) / total_count

        # "consecutive" rows for the threshold (basically shifting the X up and down visually to overlap consecutively)
        threshold = (X[1:] + X[:-1]) / 2

        best_split = np.argmin(weighted_mse)

        # return the best wmse as well as the threshold
        return weighted_mse[best_split], threshold[best_split]

        # find here the "old" for loop for the exact same functionality as above!

        # compute the mse at current node
        current_yhat = np.mean(data[target_name].mean())
        current_mse = self._calc_MSE(data[target_name], current_yhat)
        current_mse = np.round(current_mse, 3)

        # iterate over all rows of the SORTED data
        for i in range(1, len(data)):
            logger.info(data)
            logger.info(which_feature_name)
            logger.info(f"{data.iloc[i][which_feature_name]}, {data.iloc[i-1][which_feature_name]}")
            # compute average of two consecutive rows to have some threshold start 
            split_val = (data.iloc[i][which_feature_name] + data.iloc[i-1][which_feature_name]) / 2

            # split the data into the two sides
            left_branch = data[data[which_feature_name]<=split_val]
            right_branch = data[data[which_feature_name]>split_val]

            # compute the MSE of both sides
            left_yhat = np.mean(left_branch[target_name]) 
            left_mse = self._calc_MSE(left_branch[target_name], left_yhat) 

            right_yhat = np.mean(right_branch[target_name]) 
            right_mse = self._calc_MSE(right_branch[target_name], right_yhat) 

            # compute weighted MSE
            weighted_mse = ((len(left_branch) * left_mse) + (len(right_branch) * right_mse))/len(data)

            # update bestbest_weighted_mse
            if weighted_mse <= best_weighted_mse:
                best_weighted_mse = weighted_mse
                best_threshold = split_val

        return best_weighted_mse, best_threshold

    def _calc_entropy(self, target_column: str) -> np.float64:
        '''
        Calculate the entropy of a target column
       
        Parameters
        ----------
        target_column : ArrayLike
            The target column of which we want to calculate the entropy

        Returns
        -------
        np.float64
            The entropy of the given target column    
        '''

        elements, counts = np.unique(target_column, return_counts = True)

        entropy = np.sum([(-counts[i]/np.sum(counts)) * np.log2(counts[i] / np.sum(counts)) for i in range(len(elements))])
        
        return entropy
    
    def _calc_information_gain(self, data: ArrayLike, target_name:str, which_feature_name: str) -> np.float64:
        '''
        Calculate the information gain based on an attribute (with its weighted entropy!) and the target variable entropy
        
        Parameters
        ----------
        data : ArrayLike
            The data set in which we want to calculate the information gain for a given feature
        target_name: str
            Name of the target feature
        which_feature_name: str
            Name of the feature for which we want to calculate the information gain.

        Returns
        -------
        np.float64
            The infomation gain of the given column    
        '''
        target_entropy = self._calc_entropy(data[target_name])

        vals, counts = np.unique(data[which_feature_name], return_counts=True)

        # Calculate the WEIGHTED entropy
        weighted_entropy = np.sum([(counts[i] / np.sum(counts)) * self._calc_entropy(data[data[which_feature_name]==vals[i]].dropna()[target_name]) for i in range(len(vals))])
    
        # Calculate the information gain
        information_gain = target_entropy - weighted_entropy
        
        return information_gain

    def _find_best_split(self, data: pd.DataFrame, target_name: str, categorical_features:list = []) -> Dict:
        best_feature = None
        best_threshold = None
        best_variance = np.inf
        is_continuous = False

        for feature_name in data.columns:
            logger.info(f"passed categorical_features: {categorical_features}")
            if feature_name == target_name:
                continue

            elif feature_name in categorical_features:
                logger.info(f"feature {feature_name} entered categorical processing step")
                variance = self._calc_variance_cat_features(data, target_name, feature_name)
                
                # if improvement: overwrite values
                if variance <= best_variance:
                    best_feature = feature_name
                    best_threshold = None
                    best_variance = variance
                    is_continuous = False
            
            else:
                logger.info(f"feature {feature_name} entered continous processing step")

                variance, threshold = self._calc_variance_cont_features(data, target_name, feature_name)
                                
                # if improvement: overwrite values
                if variance <= best_variance:
                    best_feature = feature_name
                    best_threshold = threshold
                    best_variance = variance
                    is_continuous = True
        
        return best_feature, best_threshold, best_variance, is_continuous

    def _Classfier(
            self, 
            data: pd.DataFrame, 
            target_name: str, 
            min_instances: int = 1, 
            categorical_features: list = [], 
            max_depth: int = None, 
            features: list = None,
            depth: int = 0) -> dict[str, Any]:
        '''
        Recursive tree building algorithm.

        Parameters
        ----------
        data : pd.DataFrame
            Subset of the dataset at the current node.
        target_name : str
            Name of the target column
        min_instances : int, optional
            Minimum number of samples required to allow further splitting, else mean is taken.
        categorical_features : list, optional (but actually necessary in case of such)
            List of feature names that represent categorical features
        max_depth : int, optional
            Maximum depth of the tree. If None, the tree grows until min_instances is reached
        features : list, optional
            List of (remaining) features available for splitting
        depth : int, optional
            Depth of the recursion. Used internally

        Returns
        -------
        dict[str, Any] or float
            A nested dictionary representing the tree structure. 
            "Any" will be another such dictionary or a float (= mean target value) if the leaf node is reached

        
        '''
        # Stopping criterion for the recurrsion if we require some minimum value for the size/length of a sub dataset
        if len(data) <= min_instances:
            return np.mean(data[target_name])
    
        # If the dataset has reached the wished for tdepth, return the mean target feature value of the remaining dataset as before
        elif max_depth != None and depth >= max_depth:
            return np.mean(data[target_name])

        # Now this is the actual "tree growing" part
        else:
            # get the best split
            best_feature, best_threshold, best_variance, is_continuous = self._find_best_split(data, target_name, categorical_features)

            features = list(data.columns)
            features.remove(target_name)

            # Small fallback in case of no best value found
            if best_feature == None:
                logger.info(f"No best feature to split on found. \nDepth: {depth} \nFeatures left: {features} ")
                return np.mean(data[target_name])

            if is_continuous:
                # build the branching for continous variables
                left_branch = data[data[best_feature] <= best_threshold]
                right_branch = data[data[best_feature] > best_threshold]
                
                ''' Structure for continous feature split:
                {
                    "feature": "X1",
                    "threshold": 400,
                    "left": { subtree(s) for X <= 400 or value in case we reached conditions (ie leaf node) above},
                    "right": { subtree(s) for X > 400 or value in case we reached conditions (ie leaf node) above}
                }   
                '''
                
                # if the new split doesn't reduce the size (i.e. one of the branches has to be empty) we have to stop the recurssion
                if len(left_branch) == 0 or len(right_branch) == 0:
                    return float(np.mean(data[target_name]))
                
                if depth > 100:
                    logger.warning(f"Depth {depth}, feature {best_feature}, threshold {best_threshold}, "
                                f"left={len(left_branch)}, right={len(right_branch)}")
                return {
                    "feature": best_feature,
                    "threshold": best_threshold,
                    "left": self._Classfier(data=left_branch, 
                                            target_name=target_name, 
                                            categorical_features=categorical_features,
                                            min_instances=min_instances, 
                                            max_depth=max_depth, 
                                            depth= depth+1),
                    "right": self._Classfier(data=right_branch, 
                                             target_name=target_name, 
                                             categorical_features=categorical_features,
                                             min_instances=min_instances, 
                                             max_depth=max_depth, 
                                             depth= depth+1)
                }
            
            else:
                branches = {}
                for val in np.unique(data[best_feature]):
                    subset = data[data[best_feature] == val]

                    # If subset is the sane as parent, stop recursion as this gets stuck if the one var is always the best split
                    if len(subset) == len(data):
                        branches[val] = float(np.mean(data[target_name]))
                        continue

                    # keep this safeguard info here if we hit some weird recurssion depth issue again
                    if depth > 100:
                        logger.warning(f"Depth {depth}, value {val}, best_feature {best_feature}, lendata {len(data)}")
                    
                    branches[val] = self._Classfier(data=subset, 
                                                    target_name=target_name, 
                                                    categorical_features=categorical_features,
                                                    min_instances=min_instances, 
                                                    max_depth=max_depth, 
                                                    depth= depth+1)
                
                ''' Structure for non-continous feature split:
                {
                    "feature": "categoryXY",
                    "branches": { 
                        "X": subtree(s) for value "X" of feature categoryXY,
                        "Y": subtree(s) for value "Y" of feature categoryXY
                    }
                }   
                '''
                
                return {
                    "feature": best_feature,
                    "branches": branches
                }
    
    def fit(self, 
            data: pd.DataFrame, 
            target_name: str, 
            min_instances: int = 1,
            max_depth: int = None,
            categorical_features: list = []) -> Self:
            
        '''
        Creates ("fits") the regression tree to the given dataset.

        Parameters
        ----------
        data : pd.DataFrame
            Training dataset containing features AND the target
        target_name : str
            Name of the target column
        min_instances : int, optional
            Minimum number of samples required to allow further splitting, else mean is taken.
        max_depth : int, optional
            Maximum depth of the tree. If None, the tree grows until min_instances is reached
        categorical_features : list, optional (but actually necessary in case of such)
            List of feature names that represent categorical features

        Returns
        -------
        Self
            Stores the learned tree in `self.tree_` of the initialized instance of this class.
        '''

        self.target_name = target_name
        self.categorical_features = categorical_features
        self.max_depth = max_depth
        self.min_instances = min_instances
        self.tree_ = self._Classfier(
            data, target_name, categorical_features=categorical_features, min_instances=min_instances, max_depth=max_depth
        )
        return self

    def _predict_row(self, X_pred: ArrayLike, tree:dict, features:list, categorical_features: list = []):
        logger.info(f"Entering _predict_row with (sub)tree: {tree}")
        logger.info(f"X_pred row: {X_pred}")
        logger.info(f"categorical_features passed to _predict_row: {categorical_features}")
        
        if tree["feature"] in features:
            feature = tree["feature"]
            logger.info(f"Checking for feature: {feature}\ncategorical features passed: {categorical_features}")

            if feature in categorical_features:
                result = tree["branches"][X_pred[feature]]
            else:
                threshold = tree["threshold"]
                logger.info(f"feature: {feature} // feature_value: {X_pred[feature]} // threshold: {threshold}")
                result = tree["left"] if X_pred[feature] <= threshold else tree["right"]
            
            logger.info(f"result: {result}")
            
            # continue if not a leaf
            if isinstance(result, dict):
                return self._predict_row(X_pred, result, features, categorical_features) #.drop(feature)
            # if we get a value then return and "break" the recurssion
            else:
                if isinstance(result, (np.float64, float)):
                    return result
                else:
                    logging.error(f"result is not of type float, type: {type(result)}")
                    raise ValueError(f"result is not of type float, type: {type(result)}")

    def _predict_vectorized(self, X_pred: ArrayLike, tree:dict) -> np.ndarray:
        # return the leaf of the subset once reached!
        if not isinstance(tree, dict):
            return np.full(len(X_pred), float(tree))
        
        feature = tree["feature"]

        if "branches" in tree:
            # initialize empty result np array
            y_pred = np.empty(len(X_pred))
            for val, branch in tree["branches"].items():
                mask = X_pred[feature] == val

                # if the mask only contains False, do not go into an infinite loop!
                if not mask.any():
                    continue

                y_pred[mask] = self._predict_vectorized(X_pred[mask], branch)
            return y_pred
        
        elif "threshold" in tree:
            # initialize empty result np array
            y_pred = np.empty(len(X_pred))
            threshold = tree["threshold"]
            mask = X_pred[feature] <= threshold

            # left branch (same condition as before, only go in if we have a non-empty mask)
            if mask.any():
                y_pred[mask] = self._predict_vectorized(X_pred[mask], tree["left"])
            # right branch  
            if (~mask).any():
                y_pred[~mask] = self._predict_vectorized(X_pred[~mask], tree["right"])
            return y_pred
            
        #except Exception as e:
        #    if (len(categorical_features) == 0) and ("branches" in tree): 
        #        logger.error("Categorical feature detected and no categorical_features list passed!")
        #        raise e
        #    raise e
        
    def predict(self, X_pred: pd.DataFrame, categorical_features: list = []) -> pd.Series:
        '''
        Vectorized prediction using the fitted regression tree.

        Parameters
        -------
        X_pred : pd.DataFrame
            Input samples with the same columns used during training
        categorical_features : list, optional (but actually necessary in case of such)
            List of feature names that represent categorical features
        
        Returns
        -------

        '''
        if not isinstance(X_pred, pd.DataFrame):
            raise ValueError(f"As of now, X_pred needs to be of type pd.DataFrame. It is of type {type(X_pred)}")
        
        y_pred = self._predict_vectorized(X_pred, self.tree_)
        # to flatten the np.array and to get an output like with predict_old 
        return pd.Series(y_pred)
        
    def predict_old(self, X_pred: ArrayLike, categorical_features: list = []) -> pd.Series:
    
        '''
        "old" or rather initial implementation with row-wise .apply() execution.
        Quite a bit slower than vectorized operations (ofc)

        Parameters
        -------
        X_pred : pd.DataFrame
            Input samples with the same columns used during training
        categorical_features : list, optional (but actually necessary in case of such)
            List of feature names that represent categorical features
        
        Returns
        -------


        '''

        features = X_pred.columns
        result = X_pred.apply(self._predict_row, args=(self.tree_, features, categorical_features), axis=1)
        return result
    
    def print_tree(self, tree: dict = None, depth: int = 0) -> None:
        """
        Recursively prints the regression tree structure
        """
        indent = "  " * depth
        # initialize in first step
        if tree is None:
            tree = self.tree_

        # Continuous split
        if "threshold" in tree:
            print(f"{indent}Feature: {tree['feature']} <= {tree['threshold']}")
            if isinstance(tree["left"], dict):
                print(f"{indent}--> Left:")
                self.print_tree(tree["left"], depth + 1)
            else:
                print(f"{indent}--> Left leaf: {tree['left']:.3f}")

            print(f"{indent}Feature: {tree['feature']} > {tree['threshold']}")
            if isinstance(tree["right"], dict):
                print(f"{indent}--> Right:")
                self.print_tree(tree["right"], depth + 1)
            else:
                print(f"{indent}--> Right leaf: {tree['right']:.3f}")

        # Categorical split
        elif "branches" in tree:
            print(f"{indent}Feature: {tree['feature']} (categorical)")
            for val, branch in tree["branches"].items():
                if isinstance(branch, dict):
                    print(f"{indent}--> Value {val}:")
                    self.print_tree(branch, depth + 1)
                else:
                    print(f"{indent}--> Value {val} leaf: {branch:.3f}")
                        

## Testing Area

In [ ]:
# "test" data

# continous example
np.random.seed(0)

a, b, c = 1, 2, 3
n = 100 
x = np.linspace(-10, 10, n)  # feature values from -10 to 10
noise = np.random.normal(0, 10, n)  # some random noise
y = a * x**2 + b * x + c + noise  # quadratic equation with noise

df_continous = pd.DataFrame({'X': x, 'y': y})

# categorical examples
df_small = pd.DataFrame({'Number_of_Bedrooms':[2,2,4,1,3,1,4,2],'Price_of_Sale':[100000,120000,250000,80000,220000,170000,500000,75000]})

df = pd.read_csv("day.csv",usecols=['season','holiday','weekday','weathersit','cnt'])
df_example = df.sample(frac=0.012)

In [ ]:
rng = np.random.RandomState(1)
X = np.sort(5 * rng.rand(80, 1), axis=0)
y = np.sin(X).ravel()
y[::5] += 3 * (0.5 - rng.rand(16))

df_1 = pd.DataFrame({
    "X": X.ravel(),
    "y": y 
})
regr_1 = RegressionTreeNico()
regr_2 = RegressionTreeNico()
regr_3= RegressionTreeNico()
regr_1.fit(df_1, "y", max_depth=2)
regr_2.fit(df_1, "y", max_depth=5)
regr_3.fit(df_1, "y")


X_test = np.arange(0.0, 5.0, 0.01)[:, np.newaxis]
X_test = pd.DataFrame({"X": X_test.ravel()})
y_1 = regr_1.predict(X_test)
y_2 = regr_2.predict(X_test)
y_3 = regr_3.predict(X_test)

plt.figure()
plt.scatter(X, y, s=20, edgecolor="black", c="darkorange", label="data")
plt.plot(X_test, y_1, color="cornflowerblue", label="max_depth=2", linewidth=2)
plt.plot(X_test, y_2, color="yellowgreen", label="max_depth=5", linewidth=2)
plt.plot(X_test, y_3, color="pink", label="max_depth=inf", linewidth=2)

plt.xlabel("data")
plt.ylabel("target")
plt.title("Decision Tree Regression")
plt.legend()
plt.show()

In [ ]:
regr_1 = DecisionTreeRegressor(max_depth=2)
regr_2 = DecisionTreeRegressor(max_depth=5)
regr_3 = DecisionTreeRegressor()

regr_1.fit(X, y)
regr_2.fit(X, y)
regr_3.fit(X, y)


X_test = np.arange(0.0, 5.0, 0.01)[:, np.newaxis]
y_1 = regr_1.predict(X_test)
y_2 = regr_2.predict(X_test)
y_3 = regr_3.predict(X_test)


import matplotlib.pyplot as plt

plt.figure()
plt.scatter(X, y, s=20, edgecolor="black", c="darkorange", label="data")
plt.plot(X_test, y_1, color="cornflowerblue", label="max_depth=2", linewidth=2)
plt.plot(X_test, y_2, color="yellowgreen", label="max_depth=5", linewidth=2)
plt.plot(X_test, y_3, color="pink", label="max_depth=inf", linewidth=2)

plt.xlabel("data")
plt.ylabel("target")
plt.title("Decision Tree Regression")
plt.legend()
plt.show()

In [ ]:
testTree = RegressionTreeNico()
treee = testTree.fit(df, categorical_features=["season", "holiday", "weekday", "weathersit"], target_name="cnt")

data_encoded = pd.get_dummies(df[["season", "holiday", "weekday", "weathersit"]], drop_first=True)
sk_tree = DecisionTreeRegressor(random_state=42)
sk_tree.fit(data_encoded, df["cnt"])
y_pred = sk_tree.predict(df.drop("cnt", axis=1))

y_pred

In [ ]:
y_pred_custom = treee.predict(df.drop("cnt", axis=1))
y_pred_custom

In [ ]:
y_pred_custom_old = treee.predict_old(df.drop("cnt", axis=1), categorical_features=["season", "holiday", "weekday", "weathersit"])

In [ ]:
# check whether output of new and old prediction is the same
check = pd.DataFrame(y_pred_custom_old == y_pred_custom)
check[check[0] == False]

In [ ]:
df_continous

In [ ]:
testTree = RegressionTreeNico()
treee = testTree.fit(df_continous,target_name="y")
y_pred_custom = treee.predict(pd.DataFrame(df_continous.drop("y", axis=1)))

sk_tree = DecisionTreeRegressor(random_state=42)
sk_tree.fit(df_continous.drop("y", axis=1), df_continous["y"])
y_pred = sk_tree.predict(df_continous.drop("y", axis=1))

check = pd.DataFrame(pd.Series(y_pred) == y_pred_custom)
check[check[0] == False]

In [ ]:
# compare what happens if decoded! 
testTree = RegressionTreeNico()
sk_tree = DecisionTreeRegressor(random_state=42)

data_encoded = pd.get_dummies(df[["season", "holiday", "weekday", "weathersit"]], drop_first=True)
data_encoded2 = pd.get_dummies(
    df, 
    columns=["season", "holiday", "weekday", "weathersit"], 
    drop_first=True
)

treee = testTree.fit(df, categorical_features=["season", "holiday", "weekday", "weathersit"], target_name= "cnt")
y_pred_custom = treee.predict(pd.DataFrame(df.drop("cnt", axis=1)))

sk_tree.fit(data_encoded, df["cnt"])
y_pred = sk_tree.predict(data_encoded)

check = pd.DataFrame(pd.Series(y_pred) == y_pred_custom)
check[check[0] == False]
pd.Series(y_pred).unique()

In [ ]:
import time
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# ---- Import your class here (adjust path) ----
# from your_module import RegressionTreeNico
# If the class is in the same folder and file name is RegressionTreeNico.py:
# from RegressionTreeNico import RegressionTreeNico

# ---- Utilities ----
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def leaf_count_and_depth(tree):
    """Return (leaf_count, max_depth) for a tree represented by nested dicts or float."""
    if not isinstance(tree, dict):
        return 1, 0
    if "branches" in tree:
        leaves = 0
        maxd = 0
        for b in tree["branches"].values():
            l, d = leaf_count_and_depth(b)
            leaves += l
            maxd = max(maxd, d)
        return leaves, maxd + 1
    elif "threshold" in tree:
        l_l, d_l = leaf_count_and_depth(tree["left"])
        l_r, d_r = leaf_count_and_depth(tree["right"])
        return l_l + l_r, 1 + max(d_l, d_r)
    else:
        return 1, 0

# ---- Main benchmark ----
def run_simple_benchmark(n_samples=20000, random_state=42):
    # 1) Load dataset and subsample if requested
    data = fetch_california_housing()
    X = pd.DataFrame(data.data, columns=data.feature_names)
    y = data.target
    if n_samples < len(X):
        X, _, y, _ = train_test_split(X, y, train_size=n_samples, random_state=random_state)

    # 2) Train / val / test split (60/20/20)
    X_trainval, X_test, y_trainval, y_test = train_test_split(X, y, test_size=0.2, random_state=random_state)
    X_train, X_val, y_train, y_val = train_test_split(X_trainval, y_trainval, test_size=0.25, random_state=random_state)

    # 3) scikit-learn DecisionTreeRegressor (default settings)
    sk = DecisionTreeRegressor(random_state=random_state)
    t0 = time.perf_counter()
    sk.fit(X_train, y_train)
    sk_fit_time = time.perf_counter() - t0

    t0 = time.perf_counter()
    y_val_pred_sk = sk.predict(X_val)
    sk_predict_time_val = time.perf_counter() - t0

    sk_val_rmse = rmse(y_val, y_val_pred_sk)
    sk_leaves = sk.get_n_leaves()
    # scikit depth (approx) via tree_.max_depth
    sk_depth = getattr(sk.tree_, "max_depth", None)

    # 4) Your custom RegressionTreeNico
    # Make sure RegressionTreeNico is imported above
    rt = RegressionTreeNico()

    # prepare dataframe with target column name "target" (your fit expects a DataFrame)
    train_df = X_train.copy()
    train_df["target"] = y_train
    val_df = X_val.copy()
    val_df["target"] = y_val
    test_df = X_test.copy()
    test_df["target"] = y_test

    t0 = time.perf_counter()
    rt.fit(train_df, target_name="target", min_instances=2, max_depth=None, categorical_features=[])
    rt_fit_time = time.perf_counter() - t0

    t0 = time.perf_counter()
    # use public predict (returns pd.Series)
    y_val_pred_rt = rt.predict(X_val, categorical_features=[])
    rt_predict_time_val = time.perf_counter() - t0

    rt_val_rmse = rmse(y_val, y_val_pred_rt.values)
    rt_leaves, rt_depth = leaf_count_and_depth(rt.tree_)

    # 5) Evaluate both on test set
    t0 = time.perf_counter()
    y_test_pred_sk = sk.predict(X_test)
    sk_predict_time_test = time.perf_counter() - t0
    sk_test_rmse = rmse(y_test, y_test_pred_sk)
    sk_test_mae = mean_absolute_error(y_test, y_test_pred_sk)
    sk_test_r2 = r2_score(y_test, y_test_pred_sk)

    t0 = time.perf_counter()
    y_test_pred_rt = rt.predict(X_test, categorical_features=[])
    rt_predict_time_test = time.perf_counter() - t0
    rt_test_rmse = rmse(y_test, y_test_pred_rt.values)
    rt_test_mae = mean_absolute_error(y_test, rt_test_rmse and y_test_pred_rt) if False else mean_absolute_error(y_test, y_test_pred_rt)  # keep consistent
    rt_test_r2 = r2_score(y_test, y_test_pred_rt)

    # 6) Print summary
    print("=== Simple benchmark (no pruning) ===")
    print(f"Samples: train={len(X_train)}, val={len(X_val)}, test={len(X_test)}")
    print("\n-- scikit-learn DecisionTreeRegressor --")
    print(f"Fit time: {sk_fit_time:.3f}s, Predict time (val): {sk_predict_time_val:.4f}s, Predict time (test): {sk_predict_time_test:.4f}s")
    print(f"Val RMSE: {sk_val_rmse:.4f}, Test RMSE: {sk_test_rmse:.4f}, Test MAE: {sk_test_mae:.4f}, R2: {sk_test_r2:.4f}")
    print(f"Leaves: {sk_leaves}, Depth (sklearn tree_.max_depth): {sk_depth}")

    print("\n-- RegressionTreeNico (custom) --")
    print(f"Fit time: {rt_fit_time:.3f}s, Predict time (val): {rt_predict_time_val:.4f}s, Predict time (test): {rt_predict_time_test:.4f}s")
    print(f"Val RMSE: {rt_val_rmse:.4f}, Test RMSE: {rt_test_rmse:.4f}, Test MAE: {rt_test_mae:.4f}, R2: {rt_test_r2:.4f}")
    print(f"Leaves: {rt_leaves}, Depth: {rt_depth}")

    return {
        "sk": {
            "fit_time": sk_fit_time,
            "predict_time_val": sk_predict_time_val,
            "predict_time_test": sk_predict_time_test,
            "val_rmse": sk_val_rmse,
            "test_rmse": sk_test_rmse,
            "leaves": sk_leaves,
            "depth": sk_depth
        },
        "custom": {
            "fit_time": rt_fit_time,
            "predict_time_val": rt_predict_time_val,
            "predict_time_test": rt_predict_time_test,
            "val_rmse": rt_val_rmse,
            "test_rmse": rt_test_rmse,
            "leaves": rt_leaves,
            "depth": rt_depth
        }
    }

if __name__ == "__main__":
    # Adjust n_samples if you want a smaller or larger run
    results = run_simple_benchmark(n_samples=20000)
    print("\nReturned results dict keys:", results.keys())


# Random Forest Implementation

## Description
The Random Forest approach is based on two concepts, called bagging and subspace sampling. Bagging is the short form for *bootstrap aggregation*. Here we create a multitude of datasets of the same length as the original dataset drawn from the original dataset with replacement (the *bootstrap* in bagging). We then train a tree model for each of the bootstrapped datasets and take the majority prediction of these models for a unseen query instance as our prediction (the *aggregation* in bagging). Here we take the mean or the median for regression tree models and the mode for classification tree models.



***Sources***
https://python-course.eu/machine-learning/random-forests-in-python.php
